# Model 2: Daily Sentiment Features from News

This notebook processes news articles to generate **daily sentiment scores** using FinBERT.

**Pipeline:**
1. **Preprocess** raw news data (parse dates, clean text)
2. **Summarize** articles using LSA
3. **Score sentiment** using FinBERT (financial-domain BERT)
4. **Aggregate** to daily mean sentiment score
5. **Merge** with daily volatility to create `daily_features_for_dnn.csv`

**Output files:**
- `daily_sentiment.csv` — Daily mean sentiment scores
- `daily_features_for_dnn.csv` — Merged volatility + sentiment for future DNN

---


In [1]:
!pip install sumy

In [2]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cpu


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install seaborn

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\malvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\malvi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
import pandas as pd
from datetime import timedelta, datetime

import os
import time
from collections import defaultdict

from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.tokenizers import Tokenizer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.stemmers import Stemmer
from sumy.utils import get_stop_words

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

# preprocess

This script cleans the raw news data.

What it does:
- Load the raw CSV.
- Drop rows where Text is empty or "0".
- Normalize the Date column into one consistent UTC format.
- Keep only useful columns (Date, Url, Text, Mark).
- Sort the data by date.

Output:
news_preprocessed.csv

In [7]:
RAW_NEWS_PATH = "news_data_raw.csv"
PREPROCESSED_PATH = "news_preprocessed.csv"

def convert_to_utc(time_str: str) -> str:
    time_str = str(time_str)

    if " EDT" in time_str:
        time_str_cleaned = time_str.replace(" EDT", "")
        offset = timedelta(hours=-4)
    elif " EST" in time_str:
        time_str_cleaned = time_str.replace(" EST", "")
        offset = timedelta(hours=-5)
    else:
        offset = timedelta(hours=0)
        time_str_cleaned = time_str

    formats = [
        '%B %d, %Y — %I:%M %p',
        '%b %d, %Y %I:%M%p',
        '%d-%b-%y',
        '%Y-%m-%d',
        '%Y/%m/%d',
        '%b %d, %Y'
    ]

    for fmt in formats:
        try:
            dt = datetime.strptime(time_str_cleaned, fmt)
            if fmt == '%d-%b-%y':
                offset = timedelta(hours=0)
            dt_utc = dt + offset
            return dt_utc.strftime('%Y-%m-%d %H:%M:%S UTC')
        except ValueError:
            continue

    return "Invalid date format"


# -------- preprocess --------
news = pd.read_csv(RAW_NEWS_PATH)
print("Raw shape:", news.shape)
print(news.head())

news.columns = news.columns.str.capitalize()   # Date, Url, Text, Mark

# drop bad Text
news = news.dropna(subset=['Text'])
news = news[news['Text'].astype(str) != '0']

# convert Date
news['Date_utc_str'] = news['Date'].astype(str).apply(convert_to_utc)
news = news[news['Date_utc_str'] != "Invalid date format"].copy()

news['Date'] = pd.to_datetime(news['Date_utc_str'], utc=True)
news = news.drop(columns=['Date_utc_str'])

# sort by date
news = news.sort_values('Date', ascending=True).reset_index(drop=True)

print("After preprocess:")
print(news[['Date', 'Url', 'Mark', 'Text']].head())

news.to_csv(PREPROCESSED_PATH, index=False)
print("Saved:", PREPROCESSED_PATH)


Raw shape: (2247, 4)
                              Date  \
0  January 17, 2024 — 08:52 am EST   
1  January 17, 2024 — 08:06 am EST   
2  January 17, 2024 — 07:09 am EST   
3  January 17, 2024 — 06:00 am EST   
4  January 16, 2024 — 05:51 pm EST   

                                                 Url  \
0  https://www.nasdaq.com/articles/sp-futures-sli...   
1  https://www.nasdaq.com/articles/how-the-pieces...   
2  https://www.nasdaq.com/articles/sp-futures-tic...   
3  https://www.nasdaq.com/articles/alcoa-q4-23-ea...   
4  https://www.nasdaq.com/articles/chipmakers-lea...   

                                                Text  Mark  
0  March S&P 500 E-Mini futures (ESH24) are trend...     1  
1  Looking at the underlying holdings of the ETFs...     1  
2  March S&P 500 E-Mini futures (ESH24) are trend...     1  
3  (RTTNews) - Alcoa Corp. (AA) will host a confe...     1  
4  Market indices were lower again today (followi...     1  
After preprocess:
                       Date  

# summarize

This script creates a short summary for each news article.

What it does:
- Load news_preprocessed.csv.
- For each article, take the first few sentences as a summary.
- Store the result in a new "Summary" column.
- Save Date + Summary to news_summarized.csv.

Output:
news_summarized.csv


In [8]:
# ==============================================================================
# Summarization with LSA
# ==============================================================================
# Reduces each article to key sentences for faster/better sentiment scoring.

PREPROCESSED_PATH = "news_preprocessed.csv"
SUMMARIZED_PATH = "news_summarized.csv"

# ---- Sumy setup ----
stemmer = Stemmer("english")
summarizer = LsaSummarizer(stemmer)
tokenizer = Tokenizer("english")
summarizer.stop_words = get_stop_words("english")


def increase_weight_for_key_words(sentences, key_words):
    sentence_weights = defaultdict(float)
    for sentence in sentences:
        for word in key_words:
            if word.lower() in str(sentence).lower():
                sentence_weights[sentence] += 1
    return sentence_weights


def new_sum(text, key_words, num_sentences):
    parser = PlaintextParser.from_string(str(text), tokenizer)
    initial_summary = summarizer(parser.document, num_sentences)

    sentence_weights = increase_weight_for_key_words(parser.document.sentences, key_words)

    for sentence in initial_summary:
        sentence_weights[sentence] += 1

    final_summary = sorted(sentence_weights, key=sentence_weights.get, reverse=True)[:num_sentences]
    final_summary_text = " ".join(str(sentence) for sentence in final_summary)
    return final_summary_text


# -------- Summarize all articles --------
df = pd.read_csv(PREPROCESSED_PATH)
df.columns = df.columns.str.capitalize()

print("Preprocessed data:", df.shape)
print(df.head())

# Summarization parameters
key_words_value = set()  # Add domain keywords if desired, e.g. {"volatility", "VIX"}
num_sentences_value = 3

# Apply summarizer
df['New_text'] = df['Text'].apply(
    new_sum,
    key_words=key_words_value,
    num_sentences=num_sentences_value
)

# IMPORTANT: Keep ALL articles (Mark == 0 AND Mark == 1)
# Daily sentiment should reflect the full news landscape, not just market-moving events.
# The original filtering (Mark == 1 only) is removed.

df = df.drop(columns=['Text'])

print(f"\nSummarized {len(df)} articles")
print(f"Mark distribution:\n{df['Mark'].value_counts()}")
print(df[['Date', 'Url', 'Mark', 'New_text']].head())

df.to_csv(SUMMARIZED_PATH, index=False)
print(f"\nSaved: {SUMMARIZED_PATH}")

Preprocessed data: (1502, 4)
                        Date  \
0  2016-03-22 04:39:00+00:00   
1  2016-03-28 00:44:00+00:00   
2  2016-03-28 08:05:00+00:00   
3  2016-03-28 22:56:00+00:00   
4  2016-03-30 22:15:00+00:00   

                                                 Url  \
0  https://www.nasdaq.com/articles/pre-market-mos...   
1  https://www.nasdaq.com/articles/after-hours-mo...   
2  https://www.nasdaq.com/articles/ignore-freepor...   
3  https://www.nasdaq.com/articles/better-buy-sil...   
4  https://www.nasdaq.com/articles/alcoa-aa-to-di...   

                                                Text  Mark  
0  The NASDAQ 100 Pre-Market Indicator is down -1...     1  
1  The NASDAQ 100 After Hours Indicator is down -...     1  
2  Freeport-McMoRan is struggling mightily under ...     1  
3  Image source: Silver Wheaton.\nCommodities sto...     1  
4  Aluminum giant, Alcoa Inc.AA announced that Al...     1  

Summarized 1502 articles
Mark distribution:
Mark
1    1502
Name: count, dt

# sentiment

This step generates sentiment scores for each summarized news article
using FinBERT, a financial-domain BERT model.

What it does:
- Load news_summarized.csv (Date, Url, Mark, New_text).
- Use FinBERT to classify each news summary into:
  - negative probability
  - neutral probability
  - positive probability
- Convert these probabilities into a single sentiment score:
      sentiment_score = positive_prob − negative_prob
  (Range approximately from -1 to +1)
- Save everything into news_with_sentiment_finbert.csv.

Output:
news_with_sentiment_finbert.csv
  Columns include:
  - Date
  - Url
  - Mark
  - New_text
  - finbert_neg
  - finbert_neu
  - finbert_pos
  - sentiment_score  (main feature for Model 2)


In [9]:
# ==============================================================================
# FinBERT Sentiment Scoring + Daily Aggregation
# ==============================================================================
# This cell:
# 1. Loads summarized news articles
# 2. Scores each article using FinBERT (positive - negative probability)
# 3. Aggregates to DAILY mean sentiment score
# 4. Saves both per-article and daily sentiment CSV files

# ---------- 1. Load summarized news ----------
df = pd.read_csv("news_summarized.csv")
print("Loaded:", df.shape)
print(df.head())

# NOTE: We use ALL articles (both Mark == 0 and Mark == 1)
# to capture the full daily sentiment, not just market-moving events.
TEXT_COL = "New_text"

# ---------- 2. Load FinBERT model ----------
MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Using device:", device)


# ---------- 3. Helper: get FinBERT sentiment for one text ----------
def finbert_sentiment(text: str):
    """
    Compute FinBERT sentiment for a text.
    
    Returns:
        neg_prob, neu_prob, pos_prob, score
        where score = pos_prob - neg_prob (range ~ [-1, 1])
    """
    encoded = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding="max_length"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    neg, neu, pos = probs.tolist()
    score = pos - neg  # positive = bullish, negative = bearish
    return neg, neu, pos, score


# ---------- 4. Apply to all rows ----------
neg_list, neu_list, pos_list, score_list = [], [], [], []

for i, text in enumerate(df[TEXT_COL]):
    neg, neu, pos, score = finbert_sentiment(text)
    neg_list.append(neg)
    neu_list.append(neu)
    pos_list.append(pos)
    score_list.append(score)

    if i % 100 == 0:
        print(f"Processed {i}/{len(df)} rows...")

print(f"Finished processing {len(df)} articles.")

df["finbert_neg"] = neg_list
df["finbert_neu"] = neu_list
df["finbert_pos"] = pos_list
df["sentiment_score"] = score_list

# ---------- 5. Save per-article sentiment result ----------
out_path = "news_with_sentiment_finbert.csv"
df.to_csv(out_path, index=False)
print(f"Saved per-article sentiment: {out_path}")

# ---------- 6. Build DAILY sentiment features ----------
# This is the KEY OUTPUT for the DNN pipeline.
# Aggregate per-article sentiment to daily mean.

if not np.issubdtype(df["Date"].dtype, np.datetime64):
    df["Date"] = pd.to_datetime(df["Date"])

df["date"] = df["Date"].dt.date.astype(str)  # YYYY-MM-DD string

# Group by date: mean of all articles that day (Mark == 0 and Mark == 1)
daily_sentiment = (
    df.groupby("date")["sentiment_score"]
    .mean()
    .reset_index()
    .rename(columns={"sentiment_score": "sentiment_score_daily"})
)

# Save daily sentiment
daily_sentiment_path = "daily_sentiment.csv"
daily_sentiment.to_csv(daily_sentiment_path, index=False)
print(f"\nSaved daily sentiment: {daily_sentiment_path}")
print(f"Date range: {daily_sentiment['date'].iloc[0]} to {daily_sentiment['date'].iloc[-1]}")
print(f"Total days with news: {len(daily_sentiment)}")
print(daily_sentiment.head(10))


Loaded: (1502, 4)
                        Date  \
0  2016-03-22 04:39:00+00:00   
1  2016-03-28 00:44:00+00:00   
2  2016-03-28 08:05:00+00:00   
3  2016-03-28 22:56:00+00:00   
4  2016-03-30 22:15:00+00:00   

                                                 Url  Mark  \
0  https://www.nasdaq.com/articles/pre-market-mos...     1   
1  https://www.nasdaq.com/articles/after-hours-mo...     1   
2  https://www.nasdaq.com/articles/ignore-freepor...     1   
3  https://www.nasdaq.com/articles/better-buy-sil...     1   
4  https://www.nasdaq.com/articles/alcoa-aa-to-di...     1   

                                            New_text  
0  Over the last four weeks they have had 4 up re...  
1  The following are the most active stocks for t...  
2  That said, the big difference between the two ...  
3  Silver Wheaton posted a GAAP loss in 2015, and...  
4  The New York-based company has landed a long-t...  
Using device: cpu
Processed 0/1502 rows...
Processed 100/1502 rows...
Processed 200/15

---
## Merge with Volatility for Combined DNN

This is the final step for preparing data for the combined Model 1 + Model 2 DNN.

**Merge strategy:**
- Use `daily_volatility.csv` (from Model 1 / volatility_utils.py) as the base
- Left join with `daily_sentiment.csv`
- Fill missing sentiment with 0.0 (neutral) for days with no news
- Drop any rows with missing volatility

**Output:** `daily_features_for_dnn.csv`


In [10]:
# ==============================================================================
# 4. Merge Daily Volatility + Daily Sentiment for Future LSTM
# ==============================================================================
# IMPORTANT: Only keep sentiment dates that exist in price/volatility data!
# If a date doesn't exist in price data, it cannot be used for LSTM training.
#
# Creates: daily_features_for_dnn.csv with columns:
#   - date: YYYY-MM-DD
#   - volatility: from price data (via volatility_utils.py)
#   - sentiment_score_daily: mean FinBERT sentiment for that day

import pandas as pd

# --- Load daily volatility (computed in Model 1 via volatility_utils) ---
vol_path = "daily_volatility.csv"
vol_df = pd.read_csv(vol_path)
vol_df["date"] = pd.to_datetime(vol_df["date"]).dt.date.astype(str)
print(f"Volatility data: {len(vol_df)} rows, range: {vol_df['date'].iloc[0]} to {vol_df['date'].iloc[-1]}")

# Get set of valid price dates
valid_price_dates = set(vol_df["date"].tolist())

# --- Load daily sentiment (computed above in this notebook) ---
sent_path = "daily_sentiment.csv"
sent_df = pd.read_csv(sent_path)
sent_df["date"] = pd.to_datetime(sent_df["date"]).dt.date.astype(str)
print(f"Sentiment data (before filter): {len(sent_df)} rows")

# --- FILTER: Only keep sentiment dates that exist in price data ---
sent_df_filtered = sent_df[sent_df["date"].isin(valid_price_dates)].copy()
dropped_count = len(sent_df) - len(sent_df_filtered)
print(f"Sentiment data (after filter): {len(sent_df_filtered)} rows")
print(f"Dropped {dropped_count} sentiment rows (dates not in price data)")

# --- Merge on date ---
# Use INNER join: only dates that exist in BOTH volatility AND sentiment
# For dates with volatility but no sentiment, fill with 0.0 (neutral)
merged = vol_df.merge(sent_df_filtered, on="date", how="left")
merged["sentiment_score_daily"] = merged["sentiment_score_daily"].fillna(0.0)

# Drop any rows with NaN volatility (should not happen)
merged = merged.dropna(subset=["volatility"]).reset_index(drop=True)

# --- Summary ---
overlap_count = merged["sentiment_score_daily"].ne(0.0).sum()
print(f"\n=== Final Merged Dataset ===")
print(f"Total rows: {len(merged)}")
print(f"Days with actual sentiment data: {overlap_count}")
print(f"Days with neutral fill (no news): {len(merged) - overlap_count}")
print(f"Date range: {merged['date'].iloc[0]} to {merged['date'].iloc[-1]}")

# --- Save for future LSTM ---
merged_out_path = "daily_features_for_dnn.csv"
merged.to_csv(merged_out_path, index=False)
print(f"\nSaved: {merged_out_path}")
print(merged.head(10))


Volatility data: 2014 rows, range: 2016-02-03 to 2024-02-02
Sentiment data (before filter): 860 rows
Sentiment data (after filter): 772 rows
Dropped 88 sentiment rows (dates not in price data)

=== Final Merged Dataset ===
Total rows: 2014
Days with actual sentiment data: 772
Days with neutral fill (no news): 1242
Date range: 2016-02-03 to 2024-02-02

Saved: daily_features_for_dnn.csv
         date  volatility  sentiment_score_daily
0  2016-02-03    0.042040                    0.0
1  2016-02-04    0.047351                    0.0
2  2016-02-05    0.044904                    0.0
3  2016-02-08    0.044453                    0.0
4  2016-02-09    0.044217                    0.0
5  2016-02-10    0.044804                    0.0
6  2016-02-11    0.040135                    0.0
7  2016-02-12    0.041117                    0.0
8  2016-02-16    0.042534                    0.0
9  2016-02-17    0.041648                    0.0


---
## LSTM Sentiment Embedding

Generate vector embeddings from daily sentiment scores using LSTM.

**Approach:**
1. Create sequences of daily sentiment scores (window of N days)
2. Pass through LSTM to capture temporal patterns
3. Extract hidden state as embedding vector
4. Save embeddings for downstream DNN

**Output:** `daily_sentiment_embeddings.csv` with embedding vectors per date


In [11]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
import random
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)
random.seed(seed)
# -----------------------------
# 1. Load sentiment data
# -----------------------------
features_df = pd.read_csv("daily_features_for_dnn.csv")
features_df["date"] = pd.to_datetime(features_df["date"]).dt.date.astype(str)

sentiment_values = features_df["sentiment_score_daily"].values
dates = features_df["date"].values

# -----------------------------
# 2. Load volatility (target)
# -----------------------------
vol_df = pd.read_csv("daily_volatility.csv")
vol_df["date"] = pd.to_datetime(vol_df["date"]).dt.date.astype(str)
vol_values = vol_df.set_index("date").loc[dates]["volatility"].values

# -----------------------------
# 3. Parameters
# -----------------------------
SEQUENCE_LENGTH = 60
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
BATCH_SIZE = 32
EPOCHS = 50

# -----------------------------
# 4. Scale sentiment & volatility
# -----------------------------
sent_scaler = MinMaxScaler()
vol_scaler = MinMaxScaler()

sent_scaled = sent_scaler.fit_transform(sentiment_values.reshape(-1,1))
vol_scaled = vol_scaler.fit_transform(vol_values.reshape(-1,1))

# -----------------------------
# 5. Create sequences & targets
# -----------------------------
X_seq, y_target, target_dates = [], [], []

for i in range(SEQUENCE_LENGTH, len(sent_scaled)):
    X_seq.append(sent_scaled[i-SEQUENCE_LENGTH:i])
    y_target.append(vol_scaled[i])
    target_dates.append(dates[i])

X_seq = np.array(X_seq, dtype=np.float32)  # (n_samples, seq_len, 1)
y_target = np.array(y_target, dtype=np.float32)  # (n_samples, 1)
# ----------------------------- 
# 6. Train/val split 
# ----------------------------- 
train_ratio = 0.8 
split_idx = int(len(X_seq) * train_ratio)
X_train, X_val = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_val = y_target[:split_idx], y_target[split_idx:]
print("Train shapes:", X_train.shape, y_train.shape) 
print("Test shapes:", X_val.shape, y_val.shape) 
# ----------------------------- 
# 7. Build sentiment LSTM -> embedding -> DNN for volatility 
# ----------------------------- 
# LSTM for sentiment sequences 
inputs = Input(shape=(SEQUENCE_LENGTH, 1)) 
lstm_out = LSTM(64, return_sequences=False)(inputs) # 64-dim embedding 
embedding = lstm_out # meaningful embedding after training 
output = Dense(1)(embedding) # predict volatility 
model = Model(inputs=inputs, outputs=output) 
model.compile(optimizer='adam', loss='mse') 
# Train on X_train_sent, y_train_volatility 
model.fit( X_train, 
          y_train, 
          epochs=EPOCHS, 
          batch_size=BATCH_SIZE, 
          validation_data=(X_val, y_val), 
          callbacks=[EarlyStopping(patience=3, restore_best_weights=True)] )

# -----------------------------
# 9. Extract sentiment embeddings
# -----------------------------
embedding_model = Model(inputs=inputs, outputs=embedding)
all_embeddings = embedding_model.predict(X_seq, batch_size=BATCH_SIZE)

emb_cols = [f"sent_emb_{i}" for i in range(EMBEDDING_DIM)]
emb_df = pd.DataFrame(all_embeddings, columns=emb_cols)
emb_df["date"] = target_dates
emb_df["volatility"] = vol_scaler.inverse_transform(y_target.reshape(-1,1)).flatten()
emb_df = emb_df[["date", "volatility"] + emb_cols]

emb_df.to_csv("daily_sentiment_embeddings.csv", index=False)
print("Saved sentiment embeddings with volatility target")
print(emb_df.head())

Train shapes: (1563, 60, 1) (1563, 1)
Test shapes: (391, 60, 1) (391, 1)
Epoch 1/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 0.0213 - val_loss: 0.0089
Epoch 2/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0185 - val_loss: 0.0082
Epoch 3/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0185 - val_loss: 0.0083
Epoch 4/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0186 - val_loss: 0.0084
Epoch 5/50
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0186 - val_loss: 0.0084
62/62 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Saved sentiment embeddings with volatility target
         date  volatility  sent_emb_0  sent_emb_1  sent_emb_2  sent_emb_3  \
0  2016-04-29    0.024726    0.024915    0.004253    0.049221   -0.013081   
1  2016-05-02    0.025384    0.023009    0.009421    0.044232   -0.009451   
2  2016-05-03    0.028229    0.022085    0.011556    0.040078   -0.006664   
3  2016-05-04    0.028418    0.022215    0.010862    0.037650   -0.005184   
4  2016-05-05    0.028929   

In [12]:
# -----------------------------
# 11. Save train/test split info
# -----------------------------
train_size = split_idx
test_start_idx = SEQUENCE_LENGTH + train_size
test_dates = target_dates[train_size:]
test_start_idx = SEQUENCE_LENGTH + train_size
test_dates = target_dates[train_size:]

split_info = pd.DataFrame({
    "metric": ["train_size", "test_size", "test_start_date", "test_end_date", "sequence_length", "embedding_dim", "hidden_dim"],
    "value": [train_size, len(X_val), str(test_dates[0]), str(test_dates[-1]), SEQUENCE_LENGTH, HIDDEN_DIM, HIDDEN_DIM]
})
split_info.to_csv("sentiment_train_test_split_info.csv", index=False)
print("Saved: sentiment_train_test_split_info.csv")



Saved: sentiment_train_test_split_info.csv


LSTM training:

model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)


The LSTM sees only X_train for weight updates.

The validation_data (X_val, y_val) is never used to compute gradients or update weights. It only computes loss for monitoring/early stopping.

Embedding extraction:

all_embeddings = embedding_model.predict(X_seq, batch_size=BATCH_SIZE)


predict() does not train the model — it just runs the forward pass.

Even though you feed the entire sequence set (X_seq), the model weights are frozen, so no learning occurs.

Why loss is “there” but safe:

During LSTM training, loss is computed for both training and validation.

This does not leak into embeddings because embeddings are generated after training, and the model doesn’t see the target during predict().

✅ So your embeddings are valid for downstream DNN, no leakage, and the LSTM still used a proper loss function for training.

If you want, I can also show a diagram of data flow to make it crystal clear how train, val, embeddings, and DNN interact. It really helps prevent confusion.